In [29]:
import pandas as pd
import sqlite3
import io
from scipy.stats import chi2_contingency, binomtest

# --- 1. Data Loading and Preparation ---

# Create an in-memory SQLite database from the image data
db_connection = sqlite3.connect('validation-2.db')
db_cursor = db_connection.cursor()

# Load data from the database into a DataFrame
df_db = pd.read_sql_query("SELECT participant_number, condition, conversation_index FROM participants", db_connection)
db_connection.close()


df_csv = pd.read_csv('validation-2.csv')

# --- 2. Data Merging and Cleaning ---

# Rename columns for clarity and merging
df_csv.rename(columns={
    "Participant Id": "participant_number",
    "Which of the three roles did Forty portray?": "guessed_role"
}, inplace=True)
df_csv = df_csv.rename(columns={'Participant Id ': 'participant_number'})

# Select only the columns needed for the analysis
df_db_subset = df_db[['participant_number', 'condition', 'conversation_index']]
df_csv_subset = df_csv[['participant_number', 'guessed_role']]

# Standardize the 'guessed_role' data to match 'condition' data (lowercase, remove extra words)
df_csv_subset['guessed_role'] = df_csv_subset['guessed_role'].str.lower().str.replace(' role', '')

# Merge the two dataframes on the participant number
merged_data = pd.merge(df_db_subset, df_csv_subset, on="participant_number")

print("--- Merged Data for Analysis ---")
print(merged_data)
print("\\n" + "="*30 + "\\n")


# --- 3. Statistical Analysis ---

# Check if there's enough data to analyze
if len(merged_data) < 2:
    print("Insufficient matched data to perform meaningful statistical analysis.")
else:
    # Build a contingency table
    table = pd.crosstab(merged_data["condition"], merged_data["guessed_role"])
    print("--- Contingency Table ---")
    print(table)
    print("\n")

    # Chi-square test
    chi2, p, dof, expected = chi2_contingency(table)
    print(f"Chi² = {chi2:.3f}, p = {p:.5f}")

    if p < 0.05:
        print("→ Significant difference (95% confidence that people can tell which bot is which)")
    if p < 0.01:
        print("→ Significant difference (99% confidence that people can tell which bot is which)")

    # --- 4. Binomial Test for Accuracy ---
    # Count correct guesses
    correct = (merged_data["condition"] == merged_data["guessed_role"]).sum()
    n = len(merged_data)

    print(f"\n--- Accuracy Analysis ---")
    print(f"Correct guesses: {correct} out of {n}")

    # Observed accuracy
    p_hat = correct / n
    print(f"Observed accuracy = {p_hat:.3f}")

    # Binomial test (to check if accuracy > chance level, e.g., 1/3 if 3 possible roles)
    p_chance = 1 / merged_data["condition"].nunique()  # assumes equal probability for each condition
    binom_result = binomtest(correct, n, p_chance, alternative='greater')
    print(f"Binomial test p-value = {binom_result.pvalue:.5f}")

    # Confidence intervals
    from statsmodels.stats.proportion import proportion_confint
    ci_low, ci_high = proportion_confint(count=correct, nobs=n, alpha=0.05, method='wilson')
    ci_low_99, ci_high_99 = proportion_confint(count=correct, nobs=n, alpha=0.01, method='wilson')

    print(f"95% CI: [{ci_low:.3f}, {ci_high:.3f}]")
    print(f"99% CI: [{ci_low_99:.3f}, {ci_high_99:.3f}]")


--- Merged Data for Analysis ---
    participant_number     condition  conversation_index  guessed_role
0                    1    supportive                   1    supportive
1                    2  refutational                   1    supportive
2                    5  refutational                   2  refutational
3                    6    prebunking                   2  refutational
4                    7    supportive                   3    prebunking
5                    8  refutational                   3  refutational
6                    9    prebunking                   3    supportive
7                   10    supportive                   1    supportive
8                   11  refutational                   1  refutational
9                   12    prebunking                   1    prebunking
10                  13    supportive                   2    supportive
11                  14  refutational                   2  refutational
12                  15    prebunking        

/var/folders/xm/sz4ymg6n6_zdyg6m5_dvzrtc0000gn/T/ipykernel_72547/598946290.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_csv_subset['guessed_role'] = df_csv_subset['guessed_role'].str.lower().str.replace(' role', '')


In [31]:
# --- 3. Run Analysis for Each Conversation Index ---

# Get the unique conversation indices from the merged data
conversation_indices = sorted(merged_data['conversation_index'].unique())

for index in conversation_indices:
    print(f"--- Analysis for Conversation Index: {index} ---")
    
    # Filter the data for the current conversation index
    subset_data = merged_data[merged_data['conversation_index'] == index]
    
    # Check if there's enough data to analyze
    if len(subset_data) < 2:
        print("Insufficient matched data to perform meaningful statistical analysis for this index.\n")
        continue

    # --- Chi-Squared Test ---
    # Build a contingency table
    table = pd.crosstab(subset_data["condition"], subset_data["guessed_role"])
    print("Contingency Table:")
    print(table)
    print("")

    # Perform the test
    chi2, p, dof, expected = chi2_contingency(table)
    print(f"Chi² = {chi2:.3f}, p = {p:.5f}")

    if p < 0.05:
        print("→ Significant difference (95% confidence that people can tell which bot is which)")
    if p < 0.01:
        print("→ Significant difference (99% confidence that people can tell which bot is which)")
    print("-" * 20)

    # --- Accuracy and Confidence Intervals ---
    correct = (subset_data["condition"] == subset_data["guessed_role"]).sum()
    n = len(subset_data)
    
    if n > 0:
        p_hat = correct / n
        
        # 95% confidence interval (two-sided)
        ci_low_95, ci_high_95 = proportion_confint(count=correct, nobs=n, alpha=0.05, method='wilson')
        print(f"Correct Guesses: {correct}/{n}")
        print(f"Observed Accuracy = {p_hat:.3f}")
        print(f"95% CI: [{ci_low_95:.3f}, {ci_high_95:.3f}]")

        # 99% confidence interval
        ci_low_99, ci_high_99 = proportion_confint(count=correct, nobs=n, alpha=0.01, method='wilson')
        print(f"99% CI: [{ci_low_99:.3f}, {ci_high_99:.3f}]")
    else:
        print("No data for accuracy analysis.")
        
    print("\n" + "="*40 + "\n")

--- Analysis for Conversation Index: 1 ---
Contingency Table:
guessed_role  prebunking  refutational  supportive
condition                                         
prebunking             1             1           2
refutational           0             5           1
supportive             0             2           4

Chi² = 6.929, p = 0.13971
--------------------
Correct Guesses: 10/16
Observed Accuracy = 0.625
95% CI: [0.386, 0.815]
99% CI: [0.324, 0.853]


--- Analysis for Conversation Index: 2 ---
Contingency Table:
guessed_role  prebunking  refutational  supportive
condition                                         
prebunking             2             2           1
refutational           1             4           0
supportive             0             2           2

Chi² = 5.017, p = 0.28559
--------------------
Correct Guesses: 8/14
Observed Accuracy = 0.571
95% CI: [0.326, 0.786]
99% CI: [0.267, 0.830]


--- Analysis for Conversation Index: 3 ---
Contingency Table:
guessed_role  p

In [49]:
import pandas as pd
import sqlite3
from scipy.stats import chi2_contingency, binomtest
from statsmodels.stats.proportion import proportion_confint

# --- 1. Data Loading ---

db_connection = sqlite3.connect('validation-2.db')
df_db = pd.read_sql_query("SELECT participant_number, condition, conversation_index FROM participants", db_connection)
db_connection.close()

df_csv = pd.read_csv('validation-2.csv')

# --- 2. Clean the comment column and filter exact 'attention' values ---

comment_col = "Please provide your comments about the conversation."

# Make sure the column exists
if comment_col not in df_csv.columns:
    raise KeyError(f"Column not found: {comment_col}")

# Normalise: convert to string, strip whitespace, lowercase
df_csv['comment_clean'] = df_csv[comment_col].astype(str).str.strip().str.lower()

# Keep only rows where the cleaned comment equals exactly 'attention'
df_csv_attention = df_csv[df_csv['comment_clean'] == 'attention'].copy()

print(f"Total rows in CSV: {len(df_csv)}")
print(f"Rows EXACTLY 'attention' (after clean): {len(df_csv_attention)}")
print("Sample of matched comment_clean values (unique):", df_csv_attention['comment_clean'].unique())
print()

# Optional: drop duplicate participant rows if present (uncomment if needed)
# df_csv_attention = df_csv_attention.drop_duplicates(subset='Participant Id', keep='first')

# --- 3. Prepare columns and merge ---

# Rename guessed_role column if necessary (do this on the attention subset)
df_csv_attention = df_csv_attention.rename(columns={
    "Participant Id": "participant_number",
    "Which of the three roles did Forty portray?": "guessed_role",
    "Participant Id ": "participant_number"  # handle stray name variant
})

# Ensure participant_number column exists and has same dtype as DB
if 'participant_number' not in df_csv_attention.columns:
    raise KeyError("Participant Id column not found / renamed correctly in CSV subset.")

# Select relevant columns
df_db_subset = df_db[['participant_number', 'condition', 'conversation_index']].copy()
df_csv_subset = df_csv_attention[['participant_number', 'guessed_role']].copy()

# Clean guessed_role safely
df_csv_subset.loc[:, 'guessed_role'] = df_csv_subset['guessed_role'].astype(str).str.lower().str.replace(' role', '', regex=False).str.strip()

# Merge on participant_number (inner join)
merged_data_attention = pd.merge(df_db_subset, df_csv_subset, on="participant_number", how='inner')

print("--- Merged Data for Analysis (ATTENTION only) ---")
print(merged_data_attention)
print("\n" + "="*30 + "\n")

# --- 4. Statistical Analysis on ATTENTION subset ---

if len(merged_data_attention) < 2:
    print("Insufficient matched ATTENTION data for meaningful statistical analysis.")
else:
    table = pd.crosstab(merged_data_attention["condition"], merged_data_attention["guessed_role"])
    print("--- Contingency Table ---")
    print(table)
    print("\n")

    chi2, p, dof, expected = chi2_contingency(table)
    print(f"Chi² = {chi2:.3f}, p = {p:.5f}")

    if p < 0.05:
        print("→ Significant difference (95% confidence that people can tell which bot is which)")
    if p < 0.01:
        print("→ Significant difference (99% confidence that people can tell which bot is which)")

    correct = (merged_data_attention["condition"] == merged_data_attention["guessed_role"]).sum()
    n = len(merged_data_attention)

    print(f"\n--- Accuracy Analysis ---")
    print(f"Correct guesses: {correct} out of {n}")
    p_hat = correct / n
    print(f"Observed accuracy = {p_hat:.3f}")

    p_chance = 1 / merged_data_attention["condition"].nunique()
    binom_result = binomtest(correct, n, p_chance, alternative='greater')
    print(f"Binomial test p-value = {binom_result.pvalue:.5f}")

    ci_low, ci_high = proportion_confint(count=correct, nobs=n, alpha=0.05, method='wilson')
    ci_low_99, ci_high_99 = proportion_confint(count=correct, nobs=n, alpha=0.01, method='wilson')
    print(f"95% CI: [{ci_low:.3f}, {ci_high:.3f}]")
    print(f"99% CI: [{ci_low_99:.3f}, {ci_high_99:.3f}]")


Total rows in CSV: 45
Rows EXACTLY 'attention' (after clean): 24
Sample of matched comment_clean values (unique): ['attention']

--- Merged Data for Analysis (ATTENTION only) ---
    participant_number     condition  conversation_index  guessed_role
0                    1    supportive                   1    supportive
1                    5  refutational                   2  refutational
2                    9    prebunking                   3    supportive
3                   10    supportive                   1    supportive
4                   12    prebunking                   1    prebunking
5                   13    supportive                   2    supportive
6                   14  refutational                   2  refutational
7                   16    supportive                   3    supportive
8                   19    supportive                   1  refutational
9                   20  refutational                   1  refutational
10                  23  refutational    

In [51]:
# --- 3. Run Analysis for Each Conversation Index (ATTENTION subset) ---

# Get the unique conversation indices from the merged ATTENTION data
conversation_indices = sorted(merged_data_attention['conversation_index'].unique())

for index in conversation_indices:
    print(f"--- Analysis for Conversation Index: {index} ---")
    
    # Filter the data for the current conversation index
    subset_data = merged_data_attention[merged_data_attention['conversation_index'] == index]
    
    # Check if there's enough data to analyze
    if len(subset_data) < 2:
        print("Insufficient matched data to perform meaningful statistical analysis for this index.\n")
        continue

    # --- Chi-Squared Test ---
    # Build a contingency table
    table = pd.crosstab(subset_data["condition"], subset_data["guessed_role"])
    print("Contingency Table:")
    print(table)
    print("")

    # Perform the test
    chi2, p, dof, expected = chi2_contingency(table)
    print(f"Chi² = {chi2:.3f}, p = {p:.5f}")

    if p < 0.05:
        print("→ Significant difference (95% confidence that people can tell which bot is which)")
    if p < 0.01:
        print("→ Significant difference (99% confidence that people can tell which bot is which)")
    print("-" * 20)

    # --- Accuracy and Confidence Intervals ---
    correct = (subset_data["condition"] == subset_data["guessed_role"]).sum()
    n = len(subset_data)
    
    if n > 0:
        p_hat = correct / n
        
        # 95% confidence interval (two-sided)
        ci_low_95, ci_high_95 = proportion_confint(count=correct, nobs=n, alpha=0.05, method='wilson')
        print(f"Correct Guesses: {correct}/{n}")
        print(f"Observed Accuracy = {p_hat:.3f}")
        print(f"95% CI: [{ci_low_95:.3f}, {ci_high_95:.3f}]")

        # 99% confidence interval
        ci_low_99, ci_high_99 = proportion_confint(count=correct, nobs=n, alpha=0.01, method='wilson')
        print(f"99% CI: [{ci_low_99:.3f}, {ci_high_99:.3f}]")
    else:
        print("No data for accuracy analysis.")
        
    print("\n" + "="*40 + "\n")


--- Analysis for Conversation Index: 1 ---
Contingency Table:
guessed_role  prebunking  refutational  supportive
condition                                         
prebunking             1             1           1
refutational           0             1           0
supportive             0             2           3

Chi² = 3.600, p = 0.46284
--------------------
Correct Guesses: 5/9
Observed Accuracy = 0.556
95% CI: [0.267, 0.811]
99% CI: [0.207, 0.857]


--- Analysis for Conversation Index: 2 ---
Contingency Table:
guessed_role  prebunking  refutational  supportive
condition                                         
prebunking             1             0           0
refutational           1             2           0
supportive             0             1           2

Chi² = 6.222, p = 0.18316
--------------------
Correct Guesses: 5/7
Observed Accuracy = 0.714
95% CI: [0.359, 0.918]
99% CI: [0.278, 0.942]


--- Analysis for Conversation Index: 3 ---
Contingency Table:
guessed_role  preb